# filtrage des datasets de datagouv

In [1]:
import pandas as pd
import re


In [48]:
# lecture des datasets exportés en parquet depuis l'api datagouv dans lecture_datasets_api.ipynb
dsdatagouv = pd.read_csv("data/datasets.csv",sep=";")

In [49]:
dsdatagouv.shape

(74633, 15)

## recherche basique OU sur des mots-clés 

In [50]:
keywords = [
    "transport", "mobilité", "route", "passager", "fret", "parking", "stationnement", "voirie", "accessibilité", "enquête", "horaire", "calendrier", 
    "vélo", "trafic", "piéton", "voiture", "véhicule", "carburant", "accident", "sécurité routière", "train", "transport public", "avion", "fluvial", "maritime",
    "covoiturage", "autopartage", "bruit", "qualité de l'air"
]


In [51]:
# Construire un pattern regex : mot1|mot2|mot3...
pattern = "|".join(re.escape(kw.translate(str.maketrans("àäâéèêëûüôöïç", "aaaeeeeuuooic"))) for kw in keywords)
pattern

"transport|mobilite|route|passager|fret|parking|stationnement|voirie|accessibilite|enquete|horaire|calendrier|velo|trafic|pieton|voiture|vehicule|carburant|accident|securite\\ routiere|train|transport\\ public|avion|fluvial|maritime|covoiturage|autopartage|bruit|qualite\\ de\\ l'air"

In [52]:
import unicodedata
def normalize(text: str) -> str:
    """Minuscule + suppression des accents, pour une comparaison robuste."""
    if text is None:
        return ""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.lower()

In [53]:
dsdatagouv["c1"]=dsdatagouv["title"].apply(lambda x: normalize(x))

In [54]:
dsdatagouv["c2"]=dsdatagouv["tags"].apply(lambda x: normalize(x))

In [55]:
dsdatagouv["c3"]=dsdatagouv["description"].apply(lambda x: normalize(x))

In [56]:
cols = ["c1", "c2", "c3"]

mask = (
    dsdatagouv[cols]
    .apply(lambda s: s.str.contains(pattern, case=False, na=False, regex=True))
    .any(axis=1)
)
# filtrage des lignes contenant au moins un mot-clés dans l'une des 3 cols 
dff = dsdatagouv[mask].copy()

In [58]:
dff=dff.drop(columns=["c1", "c2", "c3"])

In [67]:
dff=dff.drop(columns=['Unnamed: 0'])

In [68]:
dff.to_csv("data/datasets_mobi.csv",sep=";")

### recherche sur les tags

on peut faire des tests de recherche par mot-clés uniquement sur les tags, les titres, les descriptions pour voir si on peut arriver à un filtrage plus pertinent
avec une liste de mots-clés basique initiale on trouve 22k datasets sur les 74k de data.gouv , on enlève déjà 2/3 des datasets et il reste sans doute beaucoup de faus positifs mais sans doute pas beaucoup de faux négatifs